# 실습 E. 정리 규칙을 코드 전체에 적용하기 (약 30분)

> **조건부 규칙을 적용하는 곳에서 조건을 누락하면 잘못된 결과가 나온다.**

코드 정리 규칙("같은 뜻의 더 나은 코드로 바꾸기")은 대부분 **어떤 조건에서만** 같은 뜻입니다. 규칙을 코드 전체에 일괄 적용하는 도구나 AI는 그 조건을 자리마다 확인하지 않을 수 있습니다.

In [1]:
import lab   # 실습 도우미: lab.run(...)이 make 명령을 실행하고 결과 보고서를 아래에 보여 줍니다

## E-1. 규칙마다 조건 적기 (10분, 직접 채우기)

이 셀을 더블클릭해서 둘째 칸을 채우세요. 셋째 칸은 E-2가 끝난 뒤 채웁니다.

| 규칙 | 같은 뜻이 되는 조건 | 이 코드에서 조건이 맞는가 |
|---|---|---|
| `list.size() == 0` → `list.isEmpty()` | | |
| 인덱스 반복 `for (int i = 0; i < list.size(); i++) { ... list.get(i) ... }` → `for (T x : list) { ... x ... }` | | |
| 변수 인라인: 한 번만 쓰는 지역 변수를 그 식으로 바꿈. `int n = e; ... n ...` → `... e ...` | | |

## E-2. 적용하고 확인하기 (15분)

세 규칙을 레거시 코드 **전체에, 적용할 수 있는 곳마다** 적용한 결과가 `scenarios/rw/<규칙>/`에 있습니다. "AI에게 이 정리 규칙을 코드 전체에 적용하라고 시킨 결과"에 해당합니다. 세 가지로 확인합니다.

1. 기존 테스트 38개
2. 기록해 둔 동작 3개: 레거시에 입력 파일 3개(`legacy/workloads/`)를 넣고, 메시지 한 줄을 처리할 때마다 허브 상태를 한 줄씩 적어 둔 기록(`golden/`)입니다. 정리한 코드에 같은 입력을 넣고 이 기록과 비교합니다
3. 무작위 입력 파일 200개로 레거시와 전체 동작 비교. 다르면 같은 차이가 나는 가장 짧은 입력을 남깁니다

In [2]:
lab.run("rewrite-check", R="IsEmpty")

$ make rewrite-check R=IsEmpty


tests: 38, passed: 38, failed: 0


기록 normal_day       같음


기록 burst            같음


기록 fridge_firmware  같음


200 random workloads (40 lines each): identical traces


In [3]:
lab.run("rewrite-check", R="ForEach")

$ make rewrite-check R=ForEach


tests: 38, passed: 38, failed: 0


기록 normal_day       같음


기록 burst            같음


기록 fridge_firmware  같음


200 random workloads (40 lines each): identical traces


In [4]:
lab.run("rewrite-check", R="InlineVariable")

$ make rewrite-check R=InlineVariable


tests: 38, passed: 38, failed: 0


기록 normal_day       같음


기록 burst            다름 (10줄)


기록 fridge_firmware  같음


MISMATCH (run 0, seed 7): minimal workload with 9 lines written to out/hubdiff-InlineVariable.txt


**변수 인라인만 동작이 바뀝니다.** 테스트는 38/38 통과하지만, 기록해 둔 동작 `burst`가 다르고 무작위 비교에서도 차이가 나옵니다. 보고서의 "바뀐 코드"에서 세 곳 중 어디가 원인인지 찾아보세요. 찾았으면 아래를 봅니다.

In [5]:
lab.diff("legacy/src/main/java/com/dxlab/homehub/dispatch/Dispatcher.java", "scenarios/rw/InlineVariable/src/main/java/com/dxlab/homehub/dispatch/Dispatcher.java", title="Dispatcher.evictOldestNonCritical: 변수 n을 인라인")

`evictOldestNonCritical`은 대기열을 한 바퀴 돌리면서 가장 오래된 비중요 이벤트 하나를 뺍니다. 레거시는 돌리기 전에 `n = queue.size()`로 횟수를 정해 둡니다. 인라인하면 반복할 때마다 `queue.size()`를 다시 읽는데, **하나를 빼는 순간 크기가 줄어 반복이 한 번 일찍 끝납니다.** 마지막 이벤트가 제자리로 돌아오지 못해 앞으로 옵니다.

```
밀어내기 전 대기열   [1, 2, 3, 4, 5, 6, 7, 8]        (8개, 모두 비중요, 중요 이벤트 11이 들어오려 함)
레거시 (n = 8)       1을 빼고 2..8을 한 바퀴 돌림 → [2, 3, 4, 5, 6, 7, 8, 11]
인라인 (size 다시 읽음) 1을 빼면 size가 7 → 2..7만 돌고 멈춤 → [8, 2, 3, 4, 5, 6, 7, 11]   ← 순서 깨짐 (규칙 R3 위반)
```

변수 인라인이 같은 뜻이 되는 조건은 "변수를 쓰는 동안 그 식의 값이 바뀌지 않는다"입니다. 다른 두 곳(`AuditLog.record`, `AuditLog.tail`)은 이 조건이 맞고, 이곳만 맞지 않습니다. 테스트는 밀어낸 개수와 빠진 이벤트만 보고 남은 순서는 보지 않아서 통과했습니다.

for-each 규칙에도 조건("반복 중에 목록을 바꾸지 않는다")이 있지만, 이 코드에서는 적용한 8곳 모두 조건이 맞아서 동작이 그대로입니다.

세 규칙을 모두 적용한 결과도 확인해 봅니다.

<details><summary><b>E-1 표의 답 보기</b></summary>

| 규칙 | 같은 뜻이 되는 조건 | 이 코드에서 조건이 맞는가 |
|---|---|---|
| `size() == 0` → `isEmpty()` | 없음(표준 컬렉션이면 늘 같음) | 3곳 모두 맞음 |
| 인덱스 반복 → for-each | 반복 중에 목록을 바꾸지 않음(그리고 인덱스를 `get`에만 씀) | 8곳 모두 맞음 |
| 변수 인라인 | 변수를 쓰는 동안 식의 값이 바뀌지 않음(식에 부작용도 없음) | `AuditLog` 두 곳은 맞음. **`Dispatcher.evictOldestNonCritical`의 `n = queue.size()`는 깨짐** |
</details>

In [6]:
lab.run("rewrite-check", R="All")

$ make rewrite-check R=All


tests: 38, passed: 38, failed: 0


기록 normal_day       같음


기록 burst            다름 (10줄)


기록 fridge_firmware  같음


MISMATCH (run 0, seed 7): minimal workload with 9 lines written to out/hubdiff-All.txt


**AI에게 직접 시켜 보기.** 아래 셀은 변수 인라인 규칙을 `Dispatcher.java`에 적용하라고 Codex에게 시킵니다. AI가 조건을 확인했습니까? `evictOldestNonCritical`의 `n`을 인라인했습니까?

In [7]:
lab.codex("P-5", 규칙="변수 인라인: 한 번만 쓰는 지역 변수를 그 식으로 바꾼다. int n = e; ... n ... → ... e ...", 코드="legacy/src/main/java/com/dxlab/homehub/dispatch/Dispatcher.java")

Codex에 로그인되어 있지 않습니다. 터미널에서 codex login 을 실행하거나, CODEX_API_KEY 환경 변수를 설정한 뒤 Jupyter를 다시 시작하세요.
Codex를 쓸 수 없으면 lab.prompt('P-5')로 프롬프트를 출력해 사내 AI 도구에 붙여 넣으세요.


## E-3. 규칙의 조건을 Dafny로 적기 (5분)

규칙마다 보조정리(lemma)가 있습니다. `..._Same`은 두 쪽이 같다는 보조정리이고, 그 `requires`가 규칙의 조건입니다. 조건부 규칙에는 조건이 없을 때 두 쪽이 다른 예(`..._Counterexample`)도 증명해 둡니다. 변수 인라인의 반례는 위의 대기열을 3개로 줄인 것입니다.

In [8]:
lab.run("rules")

$ make rules


Dafny RewriteRules.dfy                         검증 19개, 오류 0개


In [9]:
lab.show("rewrite/RewriteRules.dfy", 67, 139, title="rewrite/RewriteRules.dfy: 변수 인라인 규칙")

## 조건을 확인하면서 적용하는 도구

조건부 다시쓰기 규칙을 표현하는 도구들이 있습니다. 조건을 어디까지 표현하는지가 다릅니다.

| 도구 | 조건을 거는 방식 |
|---|---|
| Spoon (Java) | 변환을 Java 코드로 쓰고, 조건도 코드로 검사. 필요한 분석은 직접 구현 |
| Error Prone + Refaster (Java) | 바꾸기 전·후 모양을 템플릿으로 쓰고, 패턴 변수에 판별기를 붙임. 검사기에서는 null 분석 같은 값 흐름 분석 결과를 쓸 수 있음 |
| Semgrep (여러 언어) | 패턴 + 제외 조건 + 자동 수정. 메서드 안의 값 흐름 추적 |
| IntelliJ 구조 검색·바꾸기 (Java) | 패턴 변수마다 타입, 정규식, 스크립트 조건 |
| Coccinelle (C) | 의미 패치에 실행 경로 조건(`when`)을 걺 |
| Rascal, Stratego, TXL | 조건부 다시쓰기 규칙과 적용 전략을 일반적으로 표현 |

변수 인라인의 조건("쓰는 동안 값이 안 바뀜")을 확인하려면, 반복문 안에서 `queue.size()`의 값을 바꾸는 호출(`poll`)이 있는지 알아야 합니다. 값이 어디서 바뀌는지 메서드를 넘어 따라가는 **값 흐름 분석**이 필요합니다. 도구가 조건을 표현할 수 있어도, 조건의 판단은 그 분석이 얼마나 정확한가에 달려 있습니다. 그래서 적용한 뒤 동작을 비교해 확인합니다.